In [13]:
import pandas as pd

In [47]:
sales = pd.read_csv('sales.csv')
breakfast = pd.read_csv('data_breakfast_with_coordinates.csv')
lunch = pd.read_csv('data_lunch_with_coordinates.csv')

/var/folders/th/m_nrt0q54qlbtf0hq75q_wt40000gn/T/ipykernel_50679/381680460.py:2: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  breakfast = pd.read_csv('data_breakfast_with_coordinates.csv')
/var/folders/th/m_nrt0q54qlbtf0hq75q_wt40000gn/T/ipykernel_50679/381680460.py:3: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  lunch = pd.read_csv('data_lunch_with_coordinates.csv')


In [49]:
sales.columns = sales.columns.str.lower()
breakfast.columns = breakfast.columns.str.lower()
lunch.columns = lunch.columns.str.lower()

In [10]:
sales.head()

,time_of_day,school_code,school_name,date,item,description,total,free_meals,reduced_price_meals,full_price_meals,adults,alac_student,alac_adult,earned_student,earned_adult,earned_alac_student,earned_alac_adult,adj_alac,adj_meal
0,breakfast,17,COLVIN_RUN_ELEMENTARY,2025-05-01,1146,CEREAL MEAL,10,2,0,8,0,0,0,0,0,0,0,0,0
1,breakfast,17,COLVIN_RUN_ELEMENTARY,2025-05-01,1170,MINI PANCAKES,32,2,0,30,0,0,0,0,0,0,0,0,0
2,breakfast,17,COLVIN_RUN_ELEMENTARY,2025-05-01,1310,ALC BREAKFAST ENTREE,8,0,0,0,0,8,0,0,0,0,0,0,0
3,breakfast,17,COLVIN_RUN_ELEMENTARY,2025-05-01,1405,EGG & CHEESE ON BISCUIT,2,0,0,2,0,0,0,0,0,0,0,0,0
4,breakfast,17,COLVIN_RUN_ELEMENTARY,2025-05-01,151,CEREAL/ NO MILK,9,0,0,0,0,9,0,0,0,0,0,0,0


In [11]:
breakfast.head()

,school_name,date,identifier,name,planned_reimbursable,planned_non-reimbursable,planned_total,offered_reimbursable,offered_non-reimbursable,offered_total,served_reimbursable,served_non-reimbursable,served_total,discarded_total,discarded_cost,subtotal_cost,left_over_total,left_over_percent_of_offered,left_over_cost,production_cost_total
0,Aldrin Elementary,2025-05-01,10041,Mini Maple Pancakes (Package),57,0,57,59,56,0,56,21.84,0,0.0,0.0,21.84,3,5.08,1.17,23.01
1,Aldrin Elementary,2025-05-01,20001,1% White Milk (Each),33,0,33,37,34,0,34,12.24,0,0.0,0.0,12.24,3,8.11,1.08,13.32
2,Aldrin Elementary,2025-05-01,20017,Orange (6 Slices per 1/2 cup),51,0,51,55,53,0,53,18.02,0,0.0,0.0,18.02,2,3.64,0.68,18.70
3,Aldrin Elementary,2025-05-01,20038,Fat Free White Milk (Each),11,0,11,15,13,0,13,4.29,0,0.0,0.0,4.29,2,13.33,0.66,4.95
4,Aldrin Elementary,2025-05-01,30003,Honey Cheerios Cereal (Each),10,0,10,10,7,0,7,4.13,0,0.0,0.0,4.13,3,30.00,1.77,5.90


In [12]:
lunch.head()

,school_name,date,identifier,name,planned_reimbursable,planned_non-reimbursable,planned_total,offered_reimbursable,offered_non-reimbursable,offered_total,served_reimbursable,served_non-reimbursable,served_total,discarded_total,discarded_cost,subtotal_cost,left_over_total,left_over_percent_of_offered,left_over_cost,production_cost_total
0,Aldrin Elementary,2025-05-01,10044,Soft Pretzel (1 pretzel),15,0,15,31,27,0,27,5.94,0,0.0,0.0,5.94,4,12.90,0.88,6.82
1,Aldrin Elementary,2025-05-01,10062,Corn (1/4 cup (thawed)),92,0,92,155,152,0,152,16.72,0,0.0,0.0,16.72,3,1.94,0.33,17.05
2,Aldrin Elementary,2025-05-01,20001,1% White Milk (Each),44,0,44,120,43,0,43,15.48,0,0.0,0.0,15.48,77,64.17,27.72,43.20
3,Aldrin Elementary,2025-05-01,20002,Fat Free Chocolate Milk (Each),133,0,133,155,131,0,131,45.85,0,0.0,0.0,45.85,24,15.48,8.40,54.25
4,Aldrin Elementary,2025-05-01,20016,String Cheese (Each),2,0,2,5,2,0,2,0.44,0,0.0,0.0,0.44,3,60.00,0.66,1.10


In [16]:
sales['date'] = pd.to_datetime(sales['date'], errors='coerce')
breakfast['date'] = pd.to_datetime(breakfast['date'], errors='coerce')
lunch['date'] = pd.to_datetime(lunch['date'], errors='coerce')

In [17]:
def clean_numeric(df, cols):
    for col in cols:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)
                .str.replace(r"[\$,%, ]", "", regex=True)
                .replace("nan", "0")
                .astype(float)
            )
    return df

In [38]:
num_cols = [
    "served_non-reimbursable", "discarded_total", "discarded_cost",
    "subtotal_cost", "left_over_percent_of_offered", "left_over_cost",
    "left_over_total", "production_cost_total"
]
breakfast = clean_numeric(breakfast, num_cols)
lunch = clean_numeric(lunch, num_cols)

In [44]:
# --- Find popular items ---
# From sales.csv
popular_sales = (
    sales.groupby(["time_of_day", "description"])["total"]
    .sum()
    .reset_index()
    .sort_values(["time_of_day", "total"], ascending=[True, False])
)

# From breakfast_combined (by served)
popular_breakfast = (
    breakfast.groupby("name")["served_reimbursable"]
    .sum()
    .reset_index()
    .sort_values("served_reimbursable", ascending=False)
)

# From lunch_combined (by served)
popular_lunch = (
    lunch.groupby("name")["served_reimbursable"]
    .sum()
    .reset_index()
    .sort_values("served_reimbursable", ascending=False)
)

# --- Least discarded (lower is better) ---
if "discarded_total" in breakfast.columns:
    popular_breakfast_low_discarded = (
        breakfast.groupby("name")["discarded_total"]
        .sum()
        .reset_index()
        .sort_values("discarded_total", ascending=True)
    )
else:
    popular_breakfast_low_discarded = None

if "discarded_total" in lunch.columns:
    popular_lunch_low_discarded = (
        lunch.groupby("name")["discarded_total"]
        .sum()
        .reset_index()
        .sort_values("discarded_total", ascending=True)
    )
else:
    popular_lunch_low_discarded = None

# --- Least left over (lower is better) ---
if "left_over_total" in breakfast.columns:
    popular_breakfast_low_leftover = (
        breakfast.groupby("name")["left_over_total"]
        .sum()
        .reset_index()
        .sort_values("left_over_total", ascending=True)
    )
else:
    popular_breakfast_low_leftover = None

if "left_over_total" in lunch.columns:
    popular_lunch_low_leftover = (
        lunch.groupby("name")["left_over_total"]
        .sum()
        .reset_index()
        .sort_values("left_over_total", ascending=True)
    )
else:
    popular_lunch_low_leftover = None

In [45]:
# --- Results ---
print("Top 10 Breakfast Items (by served_total):")
print(popular_breakfast.head(10))

print("\nTop 10 Lunch Items (by served_total):")
print(popular_lunch.head(10))

if popular_breakfast_low_discarded is not None:
    print("\nTop 10 Breakfast Items (least discarded_total):")
    print(popular_breakfast_low_discarded.head(10))

if popular_lunch_low_discarded is not None:
    print("\nTop 10 Lunch Items (least discarded_total):")
    print(popular_lunch_low_discarded.head(10))

if popular_breakfast_low_leftover is not None:
    print("\nTop 10 Breakfast Items (least left_over_total):")
    print(popular_breakfast_low_leftover.head(10))

if popular_lunch_low_leftover is not None:
    print("\nTop 10 Lunch Items (least left_over_total):")
    print(popular_lunch_low_leftover.head(10))

print("\nTop 10 Items (from sales.csv):")
print(popular_sales.groupby("time_of_day").head(10))

# --- Export to CSV files ---
print("\n--- Exporting to CSV files ---")

# Export served_total rankings
popular_breakfast.to_csv('breakfast_popular_by_served.csv', index=False)
popular_lunch.to_csv('lunch_popular_by_served.csv', index=False)
print("Exported: breakfast_popular_by_served.csv, lunch_popular_by_served.csv")

# Export discarded_total rankings (least discarded)
if popular_breakfast_low_discarded is not None:
    popular_breakfast_low_discarded.to_csv('breakfast_popular_by_least_discarded.csv', index=False)
    print("Exported: breakfast_popular_by_least_discarded.csv")
else:
    print("Skipped: breakfast_popular_by_least_discarded.csv (column not found)")

if popular_lunch_low_discarded is not None:
    popular_lunch_low_discarded.to_csv('lunch_popular_by_least_discarded.csv', index=False)
    print("Exported: lunch_popular_by_least_discarded.csv")
else:
    print("Skipped: lunch_popular_by_least_discarded.csv (column not found)")

# Export left_over_total rankings (least leftover)
if popular_breakfast_low_leftover is not None:
    popular_breakfast_low_leftover.to_csv('breakfast_popular_by_least_leftover.csv', index=False)
    print("Exported: breakfast_popular_by_least_leftover.csv")
else:
    print("Skipped: breakfast_popular_by_least_leftover.csv (column not found)")

if popular_lunch_low_leftover is not None:
    popular_lunch_low_leftover.to_csv('lunch_popular_by_least_leftover.csv', index=False)
    print("Exported: lunch_popular_by_least_leftover.csv")
else:
    print("Skipped: lunch_popular_by_least_leftover.csv (column not found)")

Top 10 Breakfast Items (by served_total):
                                                  name  served_reimbursable
7                                   Apple Juice (Each)               422347
99                       Orange Tangerine Juice (Each)               227110
83                       Mini Maple Pancakes (Package)               196086
2                                 1% White Milk (Each)               189429
19                                       Bagel (Bagel)                92610
71                        Honey Cheerios Cereal (Each)                71032
57                                 Cream Cheese (Each)                70942
110               Red Delicious Apple (Each - serving)                69694
55   Cinnamon Toast Crunch Cereal, 25% Less Sugar (...                63964
54                         Cinnamon Chex Cereal (Each)                59553

Top 10 Lunch Items (by served_total):
                                     name  served_reimbursable
169        Fat Free 

In [41]:
# The actual amount consumed (served minus what was thrown away)

# For breakfast
breakfast_net_consumption = (
    breakfast.groupby("name")
    .agg({
        'served_reimbursable': 'sum',
        'discarded_total': 'sum'
    })
    .reset_index()
)

# Calculate net consumption
breakfast_net_consumption['net_consumption'] = (
    breakfast_net_consumption['served_reimbursable'] - breakfast_net_consumption['discarded_total']
)

# Sort by net consumption (higher is better)
breakfast_net_consumption = breakfast_net_consumption.sort_values('net_consumption', ascending=False)

# For lunch
lunch_net_consumption = (
    lunch.groupby("name")
    .agg({
        'served_reimbursable': 'sum',
        'discarded_total': 'sum'
    })
    .reset_index()
)

# Calculate net consumption
lunch_net_consumption['net_consumption'] = (
    lunch_net_consumption['served_reimbursable'] - lunch_net_consumption['discarded_total']
)

# Sort by net consumption (higher is better)
lunch_net_consumption = lunch_net_consumption.sort_values('net_consumption', ascending=False)


In [42]:
# --- Display Net Consumption Results ---
print("=== NET CONSUMPTION POPULARITY RANKINGS ===")

print("\nTop 15 Breakfast Items (by Net Consumption):")
print("=" * 90)
breakfast_display = breakfast_net_consumption[['name', 'net_consumption', 'served_reimbursable', 'discarded_total']].head(15)
breakfast_display.columns = ['Item Name', 'Net Consumption', 'Total Served', 'Total Discarded']
print(breakfast_display)

print("\nTop 15 Lunch Items (by Net Consumption):")
print("=" * 90)
lunch_display = lunch_net_consumption[['name', 'net_consumption', 'served_reimbursable', 'discarded_total']].head(15)
lunch_display.columns = ['Item Name', 'Net Consumption', 'Total Served', 'Total Discarded']
print(lunch_display)


=== NET CONSUMPTION POPULARITY RANKINGS ===

Top 15 Breakfast Items (by Net Consumption):
                                             Item Name  Net Consumption  \
7                                   Apple Juice (Each)        422247.16   
99                       Orange Tangerine Juice (Each)        227001.49   
83                       Mini Maple Pancakes (Package)        195252.97   
2                                 1% White Milk (Each)        188965.25   
19                                       Bagel (Bagel)         91235.15   
57                                 Cream Cheese (Each)         70882.48   
71                        Honey Cheerios Cereal (Each)         70799.31   
110               Red Delicious Apple (Each - serving)         69423.46   
55   Cinnamon Toast Crunch Cereal, 25% Less Sugar (...         63588.42   
54                         Cinnamon Chex Cereal (Each)         59150.37   
29                        Blueberry Chex Cereal (Each)         50500.01   
64        

In [43]:
# --- Export Net Consumption Results ---
print("=== EXPORTING NET CONSUMPTION RESULTS ===")

# Export the net consumption rankings
breakfast_net_consumption.to_csv('breakfast_net_consumption_popularity.csv', index=False)
lunch_net_consumption.to_csv('lunch_net_consumption_popularity.csv', index=False)

print("Exported: breakfast_net_consumption_popularity.csv")
print("Exported: lunch_net_consumption_popularity.csv")


=== EXPORTING NET CONSUMPTION RESULTS ===
Exported: breakfast_net_consumption_popularity.csv
Exported: lunch_net_consumption_popularity.csv


In [53]:
# Calculate leftover rate for breakfast items
breakfast_leftover_analysis = (
    breakfast.groupby("name")
    .agg({
        'left_over_total': 'sum',
        'offered_reimbursable': 'sum'
    })
    .reset_index()
)

# Calculate leftover rate (percentage)
breakfast_leftover_analysis['leftover_rate'] = (
    breakfast_leftover_analysis['left_over_total'] / 
    breakfast_leftover_analysis['offered_reimbursable'].replace(0, 1)  # Avoid division by zero
) * 100

# Sort by leftover rate (higher = more waste)
breakfast_leftover_analysis = breakfast_leftover_analysis.sort_values('leftover_rate', ascending=False)

# Calculate leftover rate for lunch items
lunch_leftover_analysis = (
    lunch.groupby("name")
    .agg({
        'left_over_total': 'sum',
        'offered_reimbursable': 'sum'
    })
    .reset_index()
)

# Calculate leftover rate (percentage)
lunch_leftover_analysis['leftover_rate'] = (
    lunch_leftover_analysis['left_over_total'] / 
    lunch_leftover_analysis['offered_reimbursable'].replace(0, 1)  # Avoid division by zero
) * 100

# Sort by leftover rate (higher = more waste)
lunch_leftover_analysis = lunch_leftover_analysis.sort_values('leftover_rate', ascending=False)


In [55]:
# --- Display Leftover Rate Results ---
print("=== LEFTOVER RATE RANKINGS ===")

print("\nTop 15 Breakfast Items (Highest Leftover Rate - Most Waste):")
print("=" * 100)
breakfast_display = breakfast_leftover_analysis[['name', 'leftover_rate', 'left_over_total', 'offered_reimbursable']].head(15)
breakfast_display.columns = ['Item Name', 'Leftover Rate (%)', 'Total Left Over', 'Total Offered']
breakfast_display['Leftover Rate (%)'] = breakfast_display['Leftover Rate (%)'].round(2)
print(breakfast_display)

print("\nTop 15 Lunch Items (Highest Leftover Rate - Most Waste):")
print("=" * 100)
lunch_display = lunch_leftover_analysis[['name', 'leftover_rate', 'left_over_total', 'offered_reimbursable']].head(15)
lunch_display.columns = ['Item Name', 'Leftover Rate (%)', 'Total Left Over', 'Total Offered']
lunch_display['Leftover Rate (%)'] = lunch_display['Leftover Rate (%)'].round(2)
print(lunch_display)


=== LEFTOVER RATE RANKINGS ===

Top 15 Breakfast Items (Highest Leftover Rate - Most Waste):
                                             Item Name  Leftover Rate (%)  \
62                      Fat Free Chocolate Milk (Each)              98.83   
37   Cereal, Cinnamon Toasters, Malt-O-Meal, IW (Su...              60.26   
64                          Fat Free White Milk (Each)              53.53   
43            Cherry Flavored Dried Cranberries (Each)              53.12   
63        Fat Free Unflavored Shelf-Stable Milk (Each)              52.70   
80   Milk, Lactose Free,  Fat Free, Shelf-Stable (E...              51.24   
96            Orange Flavored Dried Cranberries (Each)              50.00   
132   Watermelon Flavored Dried Cranberries (2 Eaches)              48.05   
17                        Assorted Cereal (Cereal Cup)              47.13   
101             Patty, Chicken, Breaded, WG (2 strips)              46.88   
66                       Frozen, Blueberries (1/2 cup)      

In [56]:
# --- Export Leftover Rate Results ---
print("=== EXPORTING LEFTOVER RATE RESULTS ===")

# Export the full leftover rate analysis
breakfast_leftover_analysis.to_csv('breakfast_leftover_rate_analysis.csv', index=False)
lunch_leftover_analysis.to_csv('lunch_leftover_rate_analysis.csv', index=False)

print("Exported: breakfast_leftover_rate_analysis.csv")
print("Exported: lunch_leftover_rate_analysis.csv")

# Calculate overall statistics
print("\n=== OVERALL LEFTOVER STATISTICS ===")
breakfast_total_leftover = breakfast['left_over_total'].sum()
breakfast_total_offered = breakfast['offered_reimbursable'].sum()
breakfast_overall_rate = (breakfast_total_leftover / breakfast_total_offered) * 100

lunch_total_leftover = lunch['left_over_total'].sum()
lunch_total_offered = lunch['offered_reimbursable'].sum()
lunch_overall_rate = (lunch_total_leftover / lunch_total_offered) * 100

print(f"Breakfast Overall Leftover Rate: {breakfast_overall_rate:.2f}%")
print(f"  Total Left Over: {breakfast_total_leftover:,.0f}")
print(f"  Total Offered: {breakfast_total_offered:,.0f}")

print(f"\nLunch Overall Leftover Rate: {lunch_overall_rate:.2f}%")
print(f"  Total Left Over: {lunch_total_leftover:,.0f}")
print(f"  Total Offered: {lunch_total_offered:,.0f}")

print(f"\nCombined Overall Leftover Rate: {((breakfast_total_leftover + lunch_total_leftover) / (breakfast_total_offered + lunch_total_offered)) * 100:.2f}%")


=== EXPORTING LEFTOVER RATE RESULTS ===
Exported: breakfast_leftover_rate_analysis.csv
Exported: lunch_leftover_rate_analysis.csv

=== OVERALL LEFTOVER STATISTICS ===
Breakfast Overall Leftover Rate: 20.26%
  Total Left Over: 496,886
  Total Offered: 2,451,946

Lunch Overall Leftover Rate: 15.78%
  Total Left Over: 1,845,563
  Total Offered: 11,695,760

Combined Overall Leftover Rate: 16.56%
